## Consolidating Employee Data

This project simulates a real-world data engineering task where employee data is scattered across multiple sources and formats. The goal was to consolidate, clean, and standardize all HR data into a single structured dataset for analysis

Objectives:
Create a single cleaned pandas DataFrame called employees_final by merging and transforming data from multiple sources:
- Excel file (employee details and emergency contacts)
- CSV file (office locations)
- JSON file (roles, salaries, teams)

I will be working with the following data in the `datasets` folder:
- __Office addresses__
    - Saved in `office_addresses.csv`.
    - Employees assigned to offices based on country
    - If the value for office is `NaN`, then the employee is remote.
- __Employee addresses__
    - Saved on the first sheet of `employee_information.xlsx`.
    - Contain personal details and address
- __Employee emergency contacts__ 
    - Saved on the second sheet of `employee_information.xlsx`; this sheet is called `emergency_contacts`.
    - Contain emergency contacts 
    - However, this sheet was edited at some point, and ***the headers were removed***! They should be: `employee_id`, `last_name`, `first_name`, `emergency_contact`, `emergency_contact_number`, and `relationship`.
- __Employee roles, teams, and salaries__ 
    - This information has been exported from the company's human resources management system into a JSON file titled `employee_roles.json`.
    - Contain job title, team, and monthly salary information

In [125]:
import pandas as pd

# Import office address CSV file
office_details = pd.read_csv('datasets/office_addresses.csv')

# Import the first Excel sheet
employee_address = pd.read_excel('datasets/employee_information.xlsx', 
                                 sheet_name=0)

# Import the second Excel sheet and add missing headers
employee_emergency = pd.read_excel('datasets/employee_information.xlsx', 
                                   sheet_name=1, 
                                   header = None,
                                   names = ['employee_id', 'last_name',
                                            'first_name', 'emergency_contact',
                                            'emergency_contact_number', 'relationship'])

# Import JSON file and make the index values columns instead of rows
employee_roles = (pd.read_json('datasets/employee_roles.json', 
             orient = 'index')
             .reset_index()
             .rename(columns={'index':'employee_id'}))

In [126]:
# Merge all four dataframes into one
employees_final = (employee_address.merge(
    office_details, 
    left_on = ['employee_country'],
    right_on = ['office_country'],
    how='left')
    .merge(
    employee_emergency,
    on='employee_id',
    how='left')
    .merge(
    employee_roles,
    on = 'employee_id',
    how = 'left')
    .set_index('employee_id'))

In [127]:
# Reorder the columns and assign to the employees_final dataframe
order = ['employee_first_name',
        'employee_last_name',
        'employee_country',
        'employee_city',
        'employee_street',
        'employee_street_number',
        'emergency_contact',
        'emergency_contact_number',
        'relationship',
        'monthly_salary',
        'team',
        'title',
        'office',
        'office_country',
        'office_city',
        'office_street',
        'office_street_number']

employees_final = employees_final[order]

In [128]:
# Change NaN values for all columns starting with office to 'Remote'
for col in employees_final.columns:
    if col.startswith('office'):
        employees_final[col] = employees_final[col].fillna('Remote')

employees_final.head()

,employee_first_name,employee_last_name,employee_country,employee_city,employee_street,employee_street_number,emergency_contact,emergency_contact_number,relationship,monthly_salary,team,title,office,office_country,office_city,office_street,office_street_number
employee_id,,,,,,,,,,,,,,,,,
A2R5H9,Jax,Hunman,BE,Leuven,Grote Markt,9,Opie Hurst,+32-456-5556-84,Brother,$4500,Leadership,CEO,Leuven Office,BE,Leuven,Martelarenlaan,38.0
H8K0L6,Tara,Siff,GB,London,Baker Street,221,Wendy de Matteo,+44-020-5554-333,Sister,$4500,Leadership,CFO,WeWork Office,GB,London,Old Street,207.0
G4R7V0,Gemma,Sagal,US,New-York,Perry Street,66,John Newmark,+1-202-555-194,Husband,$3000,Sales,Business Developer,ESB Office,US,New York City,Fifth Avenue,350.0
M1Z7U9,Tig,Coates,FR,Paris,Rue de l'Université,7,Venus Noone,+1-202-555-0130,Wife,$2000,People Operations,Office Manager,Remote,Remote,Remote,Remote,Remote
